# MFA Training Hyperparameters

This notebook trains MFA on a synthetic dataset sampled from a mixture of Gaussians. It is a self-contained demo: no external data or pretrained model is required. The synthetic data is constructed so that MFA can properly fit it (low-rank Gaussian structure per component).


In [1]:
from pathlib import Path
from types import SimpleNamespace

import torch
from torch.utils.data import DataLoader, TensorDataset

from large_scale_mfa_pipeline import make_loaders, run_from_config


## Synthetic Data Loader


In this demo we just use a set of synthetic activations sampled from a mixture of gaussians.

In [2]:
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "large_scale_mfa_pipeline.py").exists():
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_DIR = REPO_ROOT / "runs" / "notebook_demo"
RUN_NAME = "notebook_demo_mfa"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 256
SEED = 0
DATA_DTYPE = torch.float32

# Synthetic data parameters
FEATURE_DIM = 128       # dimensionality of each sample
NUM_GAUSSIANS = 16      # number of mixture components to sample from
SAMPLES_PER_GAUSSIAN = 200  # total samples = NUM_GAUSSIANS * SAMPLES_PER_GAUSSIAN
GAUSSIAN_RANK = 8       # low-rank structure in each Gaussian's covariance
VAL_FRACTION = 0.2


def create_loader(layer):
    """Return (train_loader, val_loader) with synthetic Gaussian-mixture data.

    Each mixture component has a low-rank covariance (factor * factor^T + noise*I),
    which is exactly the structure MFA is designed to model. The ``layer`` argument
    is accepted for interface compatibility with the pipeline but is unused here.
    """
    rng = torch.Generator().manual_seed(SEED + int(layer))

    chunks = []
    for k in range(NUM_GAUSSIANS):
        # Random low-rank factor matrix: (FEATURE_DIM, GAUSSIAN_RANK)
        factor = torch.randn(FEATURE_DIM, GAUSSIAN_RANK, generator=rng) * 0.5
        # Random mean in a bounded region
        mean = torch.randn(FEATURE_DIM, generator=rng) * 4.0
        # Sample noise component
        noise_std = 0.1 + torch.rand(1, generator=rng).item() * 0.4
        # Draw samples: z ~ N(0, I_rank), x = mean + factor @ z + noise
        z = torch.randn(SAMPLES_PER_GAUSSIAN, GAUSSIAN_RANK, generator=rng)
        noise = torch.randn(SAMPLES_PER_GAUSSIAN, FEATURE_DIM, generator=rng) * noise_std
        samples = mean.unsqueeze(0) + z @ factor.t() + noise
        chunks.append(samples.to(DATA_DTYPE))

    data = torch.cat(chunks, dim=0)  # (N, FEATURE_DIM)
    n = data.shape[0]

    order = torch.randperm(n, generator=rng)
    val_n = int(round(n * VAL_FRACTION))
    val_idx = order[:val_n]
    train_idx = order[val_n:]

    train_ds = TensorDataset(data[train_idx])
    val_ds = TensorDataset(data[val_idx]) if val_n else None

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=rng,
        drop_last=False,
    )
    val_loader = (
        DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
        if val_ds is not None
        else None
    )
    return train_loader, val_loader


## 1. Data Loader

The pipeline only assumes a PyTorch loader. Each batch should be either `x` or `(x, tokens_or_metadata)`, where `x` is `(batch, feature_dim)`.

`create_loader(layer)` generates a synthetic dataset drawn from a mixture of Gaussians with low-rank covariance structure — exactly the distribution MFA is designed to model. The `layer` argument is accepted for interface compatibility with the pipeline but is unused here.


In [3]:
train_loader, val_loader = create_loader(0)
(x,) = next(iter(train_loader))
print('batch shape:', x.shape)


batch shape: torch.Size([256, 128])


## 2. MFA Size

These are the most important hyper-parameters to set. In our 100 million activation runs, we saw good performance with 8K. In the end, if we want to isolate structures in single gaussians, then we use less gaussians, but if we want more accurate structures -- be able to model curved surfaces -- we need to tile them with multiple gaussians and post-hoc connect them via BFS (neighboring gaussians).

- NUM_COMPONENTS (K): number of local regions. Cost grows roughly linearly with K.
- RANK: local factor dimension in each component. Cost also grows with rank. We found rank 10 works decently, but some fine-grained features require higher rank. Also requires higher rank for better reconstruction. (ideally should be intrinsic dimension).

In [4]:
NUM_COMPONENTS = 32
RANK = 8

## 3. Runtime

- `single_gpu`: simple local run.
- `mgpu`: use Accelerate/FSDP for multi-GPU training.
- `DEVICE`: set automatically from CUDA availability.


In [5]:
TRAINING_BACKEND = "single_gpu"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## 4. Centroid Initialization

Initialization is one of the most important decisions for proper convergence. For the large scale MFAs in the paper we used projected K-Means, but our current implementation is not efficient enough for very large scale training runs. In that case I suggest using random init:

Random init means we initialize centroids to random points from the dataset, this is important since randomly sampled weights (not samples) does not converge to a good minimum.

Important knobs for K-Means:

- `KMEANS_POOL_SIZE`: sampled activations used for K-Means. Bigger is better but costs I/O and memory.
- `PROJECTED_DIM`: random projection dimension for faster K-Means. Use 128-512.
- `KMEANS_REFINE_EPOCHS`: full-dimensional refinement passes. Use 1-3 at scale.


In [12]:
INIT_METHOD = 'random' # 'random' or 'projected_kmeans'
CENTROIDS_PATH = OUTPUT_DIR / 'centroids' / 'centroids_{run_name}_L{layer}_k{num_components}.pt'
FORCE_REINIT = False

KMEANS_POOL_SIZE = 512
PROJECTED_DIM = 64
KMEANS_ITERS = 10
KMEANS_RESTARTS = 2
KMEANS_REFINE_EPOCHS = 2
KMEANS_TOL = 1e-4
KMEANS_METRIC = 'euclidean'

USE_TOKEN_WEIGHTS = False
KMEANS_SMOOTHING = 1.0
KMEANS_POWER = 1.0


## 5. MFA Parameter Initialization

These are usually safe to leave alone.

- `PSI_INIT`: initial diagonal noise variance.
- `PSI_PER_COMPONENT`: separate noise vector per component. More flexible, more parameters.
- `SCALE_INIT`: initial loading scale.
- `EPS_FLOOR`: numerical stability floor.


In [13]:
PSI_INIT = 1.0
PSI_PER_COMPONENT = False
SCALE_INIT = 1.0
EPS_FLOOR = 1e-5


## 6. Optimization

When training large scale MFAs, we used the below LR and 1 epoch over 100 million activations.

- `LR`: start near `7e-5` for large runs.
- `NUM_EPOCHS * STEPS_PER_EPOCH`: total training step budget.
- `VAL_INTERVAL`: MGPU validation/checkpoint cadence.


In [14]:
NUM_EPOCHS = 3
STEPS_PER_EPOCH = None
LR = 7e-5
GRAD_CLIP = None
LOG_INTERVAL = 10
START_EPOCH = 1
VAL_INTERVAL = 100
WANDB_PROJECT = None
RESUME_CHECKPOINT = None


## 7. Build the Config

This object is exactly what `run_training.py --config ...` provides from a Python config file. In the notebook, changing a value above is enough; for a real job, copy the same choices into `configs/example_config.py` and replace the loader there.


In [15]:
cfg = SimpleNamespace(
    OUTPUT_DIR=OUTPUT_DIR,
    RUN_NAME=RUN_NAME,
    NUM_COMPONENTS=NUM_COMPONENTS,
    RANK=RANK,
    TRAINING_BACKEND=TRAINING_BACKEND,
    DEVICE=DEVICE,
    DATA_DTYPE=DATA_DTYPE,
    SEED=SEED,
    make_loaders=create_loader,
    INIT_METHOD=INIT_METHOD,
    CENTROIDS_PATH=CENTROIDS_PATH,
    FORCE_REINIT=FORCE_REINIT,
    KMEANS_POOL_SIZE=KMEANS_POOL_SIZE,
    PROJECTED_DIM=PROJECTED_DIM,
    MODEL_VOCAB_SIZE=None,
    USE_TOKEN_WEIGHTS=False,
    KMEANS_SMOOTHING=KMEANS_SMOOTHING,
    KMEANS_POWER=KMEANS_POWER,
    KMEANS_ITERS=KMEANS_ITERS,
    KMEANS_RESTARTS=KMEANS_RESTARTS,
    KMEANS_TOL=KMEANS_TOL,
    KMEANS_METRIC=KMEANS_METRIC,
    KMEANS_REFINE_EPOCHS=KMEANS_REFINE_EPOCHS,
    PSI_INIT=PSI_INIT,
    PSI_PER_COMPONENT=PSI_PER_COMPONENT,
    SCALE_INIT=SCALE_INIT,
    EPS_FLOOR=EPS_FLOOR,
    NUM_EPOCHS=NUM_EPOCHS,
    LR=LR,
    GRAD_CLIP=GRAD_CLIP,
    LOG_INTERVAL=LOG_INTERVAL,
    STEPS_PER_EPOCH=STEPS_PER_EPOCH,
    START_EPOCH=START_EPOCH,
    VAL_INTERVAL=VAL_INTERVAL,
    WANDB_PROJECT=WANDB_PROJECT,
    RESUME_CHECKPOINT=RESUME_CHECKPOINT,
)


## 8. Build or Run

The first lines check loader construction. The final line runs the MFA training job on the real activations extracted above.


In [16]:
train_loader, val_loader = make_loaders(cfg, layer=0)
(x,) = next(iter(train_loader))
print(x.shape)

results = run_from_config(cfg)
results


torch.Size([256, 128])
[layer 0] preparing loaders
[init] sampling 32 random centroids
[init] saved centroids to /home/morg/students/ordavids1/mfa_large_scale_training/runs/notebook_demo/centroids/centroids_notebook_demo_mfa_L0_k32.pt
[layer 0] training with backend=single_gpu


Epoch 01 | Step 000010 Train NLL=431.789679: : 10it [00:00, 214.50it/s]


[epoch 01] train NLL=431.789679  val NLL=433.809497 ** best **


Epoch 02 | Step 000010 Train NLL=430.915396: : 10it [00:00, 230.83it/s]


[epoch 02] train NLL=430.915396  val NLL=432.897058 ** best **


Epoch 03 | Step 000010 Train NLL=429.966187: : 10it [00:00, 228.87it/s]


[epoch 03] train NLL=429.966187  val NLL=431.944983 ** best **
Restored best model from epoch 03 with metric=431.944983
[layer 0] done. model path: /home/morg/students/ordavids1/mfa_large_scale_training/runs/notebook_demo/models/notebook_demo_mfa_L0.ckpt


[{'layer': 0,
  'save_path': '/home/morg/students/ordavids1/mfa_large_scale_training/runs/notebook_demo/models/notebook_demo_mfa_L0.ckpt',
  'result': {'best_epoch': 3, 'best_metric': 431.9449829101562}}]

## Loading Models

Use the loader that matches how the checkpoint was written. Single-GPU training writes a plain MFA checkpoint through `modeling.mfa.save_mfa`; MGPU training writes an Accelerate/FSDP-friendly checkpoint through `modeling.model_checkpointing.save_mfa`.

Checkpoints are saved under `runs/<run_name>/models/<run_name>_L0.ckpt`. After loading, move the model to the device you want for analysis or inference and call `.eval()`.


In [17]:
from pathlib import Path
import torch

# Set this to the backend used for training: "single_gpu" or "mgpu".
trained_with = TRAINING_BACKEND

# Final checkpoint path.
checkpoint_path = OUTPUT_DIR / "models" / f"{RUN_NAME}_L0.ckpt"

if trained_with == "mgpu":
    from modeling.model_checkpointing import load_mfa
else:
    from modeling.mfa import load_mfa

model = load_mfa(str(checkpoint_path), map_location="cpu")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()
model


MFA()

## Analysis

The github has all the code for visualization, steering and interpreting components:

https://github.com/ordavid-s/decomposing-activations-local-geometry
